<h2 align="center"><b>Simulating waves on a chord</b></h2>
<div align=center>
<table style="width: 100%; border: none; border-collapse: collapse;">
<tr>

<style>
.emines-logo-wrap {
    position: relative;
    display: inline-block;
}

.emines-logo-wrap .emines-logo-sizer {
    visibility: hidden;
    display: block;
    width: auto;
}

.emines-logo-wrap .emines-logo-fill {
    position: absolute;
    inset: 0;
    background-color: #000000; /* fallback: light theme */
    -webkit-mask-image: url('../workshop_utils/assets/emines_logo.svg');
    mask-image: url('../workshop_utils/assets/emines_logo.svg');
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-size: contain;
    mask-size: contain;
    -webkit-mask-position: center;
    mask-position: center;
}
@media (prefers-color-scheme: dark) {
    .emines-logo-wrap .emines-logo-fill { background-color: #ffffff; }
}
.emines-logo-wrap .emines-logo-fill {
    background-color: var(--vscode-foreground, var(--logo-color));
}
</style>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<!-- Left side -->
<td style="width: 15%; border: none; text-align: right; vertical-align: middle;">
<div style="padding-right: 0;" class="emines-logo-wrap">
    <img class="emines-logo-sizer" src="../workshop_utils/assets/emines_logo.svg" alt="" aria-hidden="true">
    <div class="emines-logo-fill" role="img" aria-label="EMINES logo"></div>
</div>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<!-- Video -->
<td style="width: 40%; border: none; text-align: center; vertical-align: middle;">
  <video
    src="../workshop_utils/assets/wave_demo.mp4"
    autoplay
    muted
    loop
    playsinline
    width="600">
  </video>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<style>
.ephysics-logo-wrap {
    position: relative;
    display: inline-block;
}

.ephysics-logo-wrap .ephysics-logo-sizer {
    visibility: hidden;
    display: block;
    width: auto;
}

.ephysics-logo-wrap .ephysics-logo-fill {
    position: absolute;
    inset: 0;
    background-color: #000000; /* fallback: light theme */
    -webkit-mask-image: url('../workshop_utils/assets/ephysics_logo.png');
    mask-image: url('../workshop_utils/assets/ephysics_logo.png');
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-size: contain;
    mask-size: contain;
    -webkit-mask-position: center;
    mask-position: center;
}

@media (prefers-color-scheme: dark) {
    .ephysics-logo-wrap .ephysics-logo-fill { background-color: #ffffff; }
}

.ephysics-logo-wrap .ephysics-logo-fill {
    background-color: var(--vscode-foreground, var(--logo-color));
}
</style>

<!-- Right side -->
<td style="width: 15%; border: none; text-align: left; vertical-align: middle;">
<div style="padding-left: 0;" class="ephysics-logo-wrap">
    <img class="ephysics-logo-sizer" src="../workshop_utils/assets/ephysics_logo.png" alt="" aria-hidden="true">
    <div class="ephysics-logo-fill" role="img" aria-label="EPhy logo"></div>
</div>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

</tr>
</table>
</div>

<p>
In this last example, we will be simulating waves on a <b>chord with fixed ends</b>.<br>
This will require us to numerically solve the <b>Wave Equation</b>:

$$
\frac{\partial^2{u}}{\partial{t^2}} = c^2 \frac{\partial^2{u}}{\partial{x^2}}
$$

We will use <b>numpy</b> to implement calculations more efficiently.<br>
We will also use <b>matplotlib</b> to visualize graphs and animate the simulation in real time.

</p>


In [30]:
import numpy as np
import matplotlib

from workshop_utils import find_backend, export_sound
from workshop_utils import export_sound
import sys
if "matplotlib.pyplot" not in sys.modules: find_backend.use_window_backend(matplotlib)
print(f"Using Matplotlib window backend: {matplotlib.get_backend()}")

import matplotlib.pyplot as plt
from matplotlib import animation

Using Matplotlib window backend: TkAgg


<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Defining the system — Determining state variables — Choosing parameters</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

Our physical system is a **chord** with length $\bm{L}$ and linear mass $\bm{\mu}$ pulled with tension $\bm{T}$.\
We are studying the evolution of the displacement $\bm{u(x, t)}$ with respect to both **position** and **time**.\
The displacement by itself is not enough to fully describe the state of the chord. It's also important to consider it's **time derivative**.\
The full state is therefore $(\bm{u(x,t)}, \bm{\frac{\partial u(x,t)}{\partial t}})$.

**Other parameters you can control:**
- During the animation, the simulation will be slowed down by a factor of `timescale`.
- The simulation will stop at `duration` seconds.
- `Nx` is the amount of points along the space axis.
- `Nt` is the amount of points along the time axis.

**Helpful parameters we will calculate using previous ones:**
- `c` is the wave propagation speed in the chord.
- `K` is the squared Courant's number, keep it < 1.
- `f1` is the fundamental frequency of the chord.
- `dx` is the spacial step used in the approximation.
- `dt` is the timestep.

Make sure you can handle the required memory.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will calculate the spacial step <b>dx</b> and the time step <b>dt</b>.
</span></div>
<br>

***

In [29]:
L   = 1.0        # chord length (m)
T   = 120.0      # tension (N)
mu  = 0.005      # linear mass density (kg/m)

time_scale = 250 # Animation slow-down factor
duration = 0.1

Nx = 1000
Nt = 20_000

dx = L / (Nx-1) ##
dt = duration / (Nt-1) ##

c = np.sqrt(T/mu)
K = (c * dt / dx)**2
f1 = c / (2 * L)

print(
f'''
Wave speed: {c = :.2f}
Courant number: {np.sqrt(K) = :.2f}
Fundamental frequency: {f1 = :.2f}
Number of spacial intervals: {Nx = }
Number of time states: {Nt = }
Memory required: {Nx * Nt * 8 // 1e6} MB
''')


Wave speed: c = 154.92
Courant number: np.sqrt(K) = 0.77
Fundamental frequency: f1 = 77.46
Number of spacial intervals: Nx = 1000
Number of time states: Nt = 20000
Memory required: 160.0 MB



<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Initial conditions</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

Unlike previous examples, our state $(\bm{u(x,t)}, \bm{\frac{\partial u(x,t)}{\partial t}})$ is not just a scalar or a finite set of scalars.\
Here, our state at time $\bm{t}$ is a set of two **functions** that take position on the chord as input and outputs the corresponding displacement and velocity.


Therefore, the initial condition is also a set of two functions
$
\begin{aligned}
f &: [0,L] \to \mathbb{R}\\
\space x &\mapsto u(x,0)
\end{aligned}
$
and
$
\begin{aligned}
g &: [0,L] \to \mathbb{R}\\
\space x &\mapsto \frac{\partial u(x,t)}{\partial t}\Big|_{t=0}
\end{aligned}
$

Below, you can choose an initial condition, or make your own!\
$f$ and $g$ take $[0, L]$ as input. $f$ outputs initial displacement while $g$ outputs initial velocity at time $t=0$.\
Make sure they output a `float`. for example use `0.` instead of `0` to force output to be a float.

**Included examples:**
- **One wave:** The chord is fully idle except for a sine-shaped wave centered at `initial_pos`.
- **Pluck:** Simulates plucking a string at `x=peak` without giving an initial speed.
- **Hammer:** Simulates hitting a fully idle chord with a hammer at `hammer_speed` $m/s$
- **Standing:** Standing waves with configurable number of `knots`

The cell also contains code to plot $f$ and $g$ using **matplotlib**.

For the **boundary conditions**, we will be using **fixed ends**. This corresponds to: $\bm{\forall t>0; u(0, t) = u(L, t) = 0}$\
We will implement them later after the discretization.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will <b>choose</b> an initial condition from the provided list or come up with yours.<br>
You will also implement code to <b>plot</b> the initial condition.
</span></div>
<br>

***

In [ ]:
# --- One wave --- #

# amplitude = 0.01
# width = 0.2
# initial_pos = 0.5

# def f(x):
#     return amplitude * np.sin( (x- (initial_pos-width/2) ) * np.pi / width)**2 if np.abs(x-initial_pos) < width/2 else 0.

# def g(x):
#     return c * amplitude * np.pi / width * -np.sin( (x- (initial_pos-width/2) ) * 2 * np.pi / width) if np.abs(x-initial_pos) < width/2 else 0.

# --- Pluck --- #

amplitude = 0.01
peak = 0.2

def f(x):
    return x * amplitude/peak if x < peak else (x-peak) * -amplitude/(L-peak) + amplitude
def g(x):
    return 0.

# --- Hammer --- #

# amplitude = 0.005
# hammer_pos = 0.5
# hammer_width = 0.05
# hammer_speed = 50

# def f(x):
#     return 0.

# def g(x):
#     return hammer_speed if abs(x - hammer_pos) < hammer_width/2 else 0.

# --- Standing --- #

# amplitude = 0.01
# knots = 4

# def f(x):
#     return amplitude * np.sin(x * (knots-1) * np.pi / L)
# def g(x):
#     return 0.

# --- Make you own --- # (optional)

...

# --- Plotting --- #

x_array_temp = np.linspace(0, L, Nx)
f_array_temp = np.vectorize(f)(x_array_temp)
g_array_temp = np.vectorize(g)(x_array_temp)

fig, (ax_f, ax_g) = plt.subplots(2, 1, figsize=(8, 5))

ax_f.plot(x_array_temp, f_array_temp)
ax_f.set_ylabel("displacement (m)")
ax_f.set_title("Initial condition  u(x, 0) = f(x)")
ax_f.grid(alpha=0.3)

ax_g.plot(x_array_temp, g_array_temp)
ax_g.set_xlabel("x (m)")
ax_g.set_ylabel("velocity (m/s)")
ax_g.set_title("Initial condition  ∂u/∂t(x, 0) = g(x)")
ax_g.grid(alpha=0.3)

fig.tight_layout()
plt.show()

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>System discretization — Boundary conditions</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

Our discretized system can be thought of as a grid. The **vertical** dimension represents **space**, the **horizontal** one represents **time**.

The approximated value of the displacement at position index `i` and time index `j` is `u[i, j]` = $\bm{u(i \times \mathrm{dx}, \ j \times \mathrm{dt})}$\
For the array bounds: `u[Nx-1, Nt-1]` = $\bm{u((N_x-1) \times \mathrm{dx}, \ (N_t-1) \times \mathrm{dt})}$ = $\bm{u(\mathrm{L}, \ \mathrm{duration})}$

Note that keeping the whole grid in memory is inefficient. If history is not needed, you may only need to store enough rows required for calculations.\
For simplicity, we will keep using the inefficient grid model.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will <b>initialize</b> the grid array and <b>update</b> it with initial and boundary conditions.
</span></div>
<br>

***

In [ ]:
def initialize_simulation(L: float,
                          Nx: int,
                          Nt: int,
                          f,
                          g
                          ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    u = np.empty(shape=(Nx, Nt), dtype=np.float64)

    x_array = np.linspace(0, L, Nx)
    f_array = np.vectorize(f)(x_array)
    g_array = np.vectorize(g)(x_array)

    u[:, 0] = f_array

    u[0, :] = u[-1, :] = 0.

    return u, x_array, f_array, g_array

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Differential equation discretization</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

Our goal now is to **discretize derivatives** using differences of displacements at nearby points.

We begin by approximating $\mathrm{u(x,t+dt)}$ and $\mathrm{u(x,t-dt)}$ using second-order Taylor series:

$$
\begin{cases}
u(x,t+dt)
\approx u(x,t) + \frac{\partial u}{\partial t}(x,t)\,dt
+ \frac{1}{2}\frac{\partial^2 u}{\partial t^2}(x,t)\,dt^2
\\[1em]
u(x,t-dt)
\approx u(x,t) - \frac{\partial u}{\partial t}(x,t)\,dt
+ \frac{1}{2}\frac{\partial^2 u}{\partial t^2}(x,t)\,dt^2
\end{cases}
$$


This leads to an approximation of $\mathrm{\frac{\partial^2 u}{\partial{t^2}}}$ using only nearby grid points,
we can do a similar work for $\mathrm{\frac{\partial^2 u}{\partial{x^2}}}$:

$$
\begin{cases}
\frac{\partial^2 u}{\partial t^2}(x,t)
=
\frac{1}{dt^2}
\left(
u(x,t+dt)+u(x,t-dt)-2u(x,t)
\right)
\\[1em]
\frac{\partial^2 u}{\partial x^2}(x,t)
=
\frac{1}{dx^2}
\left(
u(x+dx, t)+u(x-dx, t)-2u(x, t)
\right)
\end{cases}
$$

After substituting in the wave equation $\frac{\partial^2{u}}{\partial{t^2}} = c^2 \frac{\partial^2{u}}{\partial{x^2}}$ , we can predict the future!

$$
u(x,t+dt)
=
2u(x,t)-u(x,t-dt) 
+
\left(\frac{c\,dt}{dx}\right)^2
\left(
u(x+dx, t)+u(x-dx, t)-2u(x, t) 
\right)
$$

In code, this means: $(\forall i \in [\![1,N_x-2]\!] \ ) \ (\forall j \in [\![1,N_t-2]\!] \ );$ `u[i, j+1] = 2 * u[i, j] - u[i, j-1] + K * (u[i+1, j] + u[i-1, j] - 2 * u[i, j])`

Spacial boundary values at `i=0` and `i=Nx-1` are taken care of by the boudary conditions implemented previously.

But we still can't calculate the time row `j=1` since the formula won't work for `j=0` because `u[i, -1]` does not exist.

However, using $ g(x) = \frac{\partial u}{\partial t}(x, 0) $ , we can prove:

$$
u(x,dt)
=
u(x,0)+g(x)dt
+
\frac12
\left(\frac{c\,dt}{dx}\right)^2
\left(
u(x+dx, 0)+u(x-dx, 0)-2u(x, 0) 
\right)
$$

Which corresponds to: $(\forall i \in [\![1,N_x-2]\!] \ ); $ `u[i, 1] = u[i, 0] + g_array[i] * dt + K/2 * (u[i+1, 0] + u[i-1, 0] - 2 * u[i, 0])`

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will program a <b>vectorized implementation</b> of the formulae above.
</span></div>
<br>

***

In [ ]:
def advance_simulation(K: float,
                       dt: float,
                       g_array: np.ndarray,
                       u: np.ndarray,
                       j: int):
    
    if j < 0: raise ValueError("time index must be positive")

    if j == 0: u[1:-1, 1] = u[1:-1, 0] + g_array[1:-1] * dt + K/2 * (u[2:, 0] + u[:-2, 0] - 2 * u[1:-1, 0])
    
    if j >= 1: u[1:-1, j+1] = 2 * u[1:-1, j] - u[1:-1, j-1] + K * (u[2:, j] + u[:-2, j] - 2 * u[1:-1, j])

    return

# --- Test implementation --- #

u_temp, x_array_temp, f_array_temp, g_array_temp = initialize_simulation(L, Nx, Nt, f, g)

for j in range(0, 100): advance_simulation(K, dt, g_array_temp, u_temp, j)

plt.plot(x_array_temp, u_temp[:, 0])
plt.plot(x_array_temp, u_temp[:, 50])
plt.plot(x_array_temp, u_temp[:, 100])
plt.show()
u_temp = x_array_temp = f_array_temp = g_array_temp = None # to free memory

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Animation of the simulation</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

Now we have everything we need to run the simulation. We just need to visualize it!

We will now implement a real-time visualization using the class `animation.FuncAnimation` from matplotlib.\
This class creates an animation by repeatedly calling a given function `func` each `interval` ms and redrawing the given figure `fig` until `frames` frames are drawn.\
Since `fig` is redrawn each frame, we can make `func` update the `fig`'s plot to match the latest state.

<br>

We begin by creating a figure `fig` and draw the first `line` (plot) that corresponds to the initial condition $f$.\
The `update` function should:
- Calculate the time indicies corresponding to the previous and next frames
- Advance the simulation between the previous frame and the new frame.
- Update the `line` with data corresponding to the new frame.
- Update the `title` of the figure to match the current progress.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will implement the requried <b>update</b> function after <b>calculating</b> some parameters.
</span></div>
<br>

***

In [ ]:
fps = 30
advance_per_frame = 1 / (time_scale * fps * dt)
n_frames = int(fps * duration * time_scale)
ymax = 1.3 * amplitude

u, x_array, f_array, g_array = initialize_simulation(L, Nx, Nt, f, g)

fig, ax = plt.subplots(figsize=(8, 3))
line, = ax.plot(x_array, f_array)
ax.set_xlim(0, L)
ax.set_ylim(-ymax, ymax)
ax.set_xlabel("x (m)")
ax.set_ylabel("displacement (m)")
ax.axhline(0, color="k", lw=0.5)
title = ax.set_title("")

def update(k):

    old_frame = int(advance_per_frame * k)
    new_frame = int(advance_per_frame * (k+1))

    for j in range(old_frame, new_frame): advance_simulation(K, dt, g_array, u, j)

    line.set_ydata(u[:, new_frame])
    title.set_text(f"t = {new_frame * dt * 1e3:.2f} ms   (slowed {time_scale}x)")

    return

anim = animation.FuncAnimation(fig=fig, func=update, frames=n_frames, interval=1000 / fps, repeat=False)

plt.show()
u=None

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Optional: Generate a sound file from the simulation</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

You can run the cell below to produce a sound file that corresponds to your parameters and initial conditions.\
The imported function `generate_sound` records the amplitude at the position `pickup_location` on the chord.\
It then applies exponential decay to simulate damping, and finally writes the results to a sound file.

You can go back to the second cell and choose different intial conditions to see how they sound like.\
We recommend leaving the other function arguments as they are.

We also recommend installing `numba` in the venv to make the sound production faster.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Run the cell below create a sound file.
</span></div>
<br>

***

In [ ]:
time_array, pickup_array = export_sound.generate_sound(K=0.8, L=1.0, Nx=500, Nt=1_000_000, duration=3, f=f, g=g, pickup_location=0.15, damping=1.)

plt.plot(time_array, pickup_array)
plt.show()

from IPython.display import Audio
Audio("wave_output.wav")